In [1]:
import numpy as np
import random
import time
from numba import jit
import matplotlib.pyplot as plt


@jit(nopython=True)
def neighbor(N, k, di):
    neigh = np.zeros((N, 4))
    for i in range(N):
        for j in range(K):
            it = ((i + di[j]) % N + N) % N  # periodic boundary condition
            neigh[i][j] = it
    return neigh

# Reward Matrix
@jit(nopython=True)
def PayoffFunction(A1, A2, x):
    if A1 != A2:
        if A1 == 1:  # CD
            off = -x
        else:
            off = 1 + x
    else:
        if A1 == 1:  # CC
            off = 1
        else:
            off = 0
    return off

@jit(nopython=True)
def reputation(N, Action, Re, c, c_max):
    for i in range(N):
        if Action[i] == 1:
            Re[i] = Re[i] + c
        else:
            Re[i] = Re[i] - c
    Re[Re > c_max] = c_max
    Re[Re < 0] = 0
    return Re

@jit(nopython=True)
def PD_Q_game(N, epsilon, gamma, alpha, Max_t, L, x, c, c_max):
    percent = np.zeros(Max_t)
    Repu = np.zeros(Max_t)
    Q_table = np.random.randn(N, 6, 2)
    Action = np.random.randint(0, 2, N)
    State = np.zeros(N)
    Reward = np.zeros(N)
    State_new = np.zeros(N)
    di = [1, -1, L, -L]
    Re = np.random.random(N)
    neig = neighbor(N, K, di)
    Fitness = np.zeros(N)
    
# initialization state
    for i in range(N):
        itC_num = 0
        for j in neig[i, :]:
            itC_num += Action[int(j)]
        State[i] = Action[i] + itC_num

    for t in range(0, Max_t):

        for i in range(N):
            p = random.random()
            if p < epsilon:
                Action[i] = random.randint(0, 1)
            else:
                Action[i] = Q_table[i][int(State[i]), :].argmax()

        C_num = np.sum(Action)
        for i in range(0, N):
            Payoff = 0
            itC_num = 0
            for j in neig[i]:
                itC_num += Action[int(j)]
                A1 = int(Action[i])
                A2 = int(Action[int(j)])
                off = PayoffFunction(A1, A2, x)
                Payoff = Payoff + off
            Reward[i] = Payoff / 4
            State_new[i] = Action[i] + itC_num

        Re = reputation(N, Action, Re, c, c_max)
        for i in range(N):
            repu = 0
            Reward_neighbor = 0
            for j in neig[i]:
                repu = repu + Re[int(j)]
                Reward_neighbor += Reward[int(j)]
            repu = repu * 0.25
            Fitness[i] = (1 - repu) * Reward[i] + repu * Reward_neighbor * 0.25
            Qmax_new = Q_table[i][int(State_new[i]), :].max()
            Q_table[i][int(State[i])][int(Action[i])] = (1 - alpha) * Q_table[i][int(State[i])][
                int(Action[i])] + alpha * (Fitness[i] + gamma * Qmax_new)

        State = np.copy(State_new)
        percent[t] = C_num / N
        Repu[t] = np.sum(Re)/N
    return percent, Repu

In [ ]:
if __name__ == '__main__':                              
    start_time = time.time()                            # Start time
#     x = 0.1                                             # Game parameters
    N = 2500                                           # Size of population
    L = 50                                             # System size
    Max_t = 200000000                                     # Time
    epsilon = 0.01                                      # E-greedy
    K = 4                                               # Number of neighbors
    alpha = 0.1
    gamma = 0.9                                         # Discount factory
    c = 0.2                                            # Reputation
    x = 0.425
    for i in range(10):
        fc, Re = PD_Q_game(N, epsilon, gamma, alpha, Max_t, L, x, c,c_max=1)
        t_log = int(pow(10, 3))
        t_log_exp = 3
        name = str(i) + 'Rmax=1_Re_b='+str(x)+'_c='+ str(c) +'_gamma='+ str(gamma) +'_alpha=' + str(alpha) + '.txt'
        f = open(name, 'w')
        for t in range(Max_t):
            if t <= 10:
                print(t, fc[t], Re[t], file=f)
            if t > 10 and t < 1000 and t % 10 == 0:
                print(t, fc[t], Re[t], file=f)
            else:
                if t == t_log:
                    print(t, fc[t], Re[t], file=f)
                    t_log_exp += 0.0025
                    t_log = int(pow(10, t_log_exp))
        f.close()
        end_time = time.time()
        print(f"spend time {end_time - start_time}")        # End time
